# 05 - Neural Network\n\nRede neural em PyTorch sobre uma amostra da silver. O objetivo aqui e capacidade preditiva, nao throughput distribuido.

In [ ]:
import time\nfrom pathlib import Path\n\nimport matplotlib.pyplot as plt\nimport numpy as np\nimport pandas as pd\nimport seaborn as sns\nimport torch\nfrom sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score\nfrom sklearn.model_selection import train_test_split\nfrom sklearn.preprocessing import StandardScaler\nfrom pyspark.sql import SparkSession, functions as F\nfrom torch import nn\nfrom torch.utils.data import DataLoader, TensorDataset\n\nSEED = 42\nSPLIT_DATE = '2023-06-01'\nSILVER_PATH = '/data/silver/trips_silver'\nRESULTS_PATH = Path('/results/model_comparison.csv')\nMODEL_OUTPUT = Path('/models/neural_network.pt')\nRESIDUAL_PLOT = Path('/results/residual_analysis_neural_network.png')\nTARGET_COL = 'base_passenger_fare'\n\nNUMERIC_COLS = [\n    'trip_miles', 'trip_time', 'wait_time_sec', 'speed_mph', 'pickup_hour',\n    'pickup_dow', 'pickup_month_num', 'is_weekend', 'is_rush_hour',\n    'is_late_night', 'pickup_airport', 'dropoff_airport', 'same_borough',\n    'shared_req', 'wav_req'\n]\nCATEGORICAL_COLS = ['hvfhs_license_num', 'pu_borough', 'do_borough']\nRAW_CONTEXT_COLS = ['pu_borough', 'trip_miles', TARGET_COL]\nSAMPLE_COLS = list(dict.fromkeys(RAW_CONTEXT_COLS + NUMERIC_COLS + CATEGORICAL_COLS))\n\nTRAIN_FRACTION = 0.01\nTEST_FRACTION = 0.02\nMAX_TRAIN_ROWS = 250_000\nMAX_TEST_ROWS = 100_000\nBATCH_SIZE = 1024\nEPOCHS = 30\nPATIENCE = 5\nLEARNING_RATE = 1e-3\n\ntorch.manual_seed(SEED)\nnp.random.seed(SEED)\ndevice = torch.device('cuda' if torch.cuda.is_available() else 'cpu')\n\nspark = (SparkSession.builder\n    .appName('nyc-rideshare-neural-network')\n    .master('spark://spark-master:7077')\n    .config('spark.executor.memory', '3g')\n    .config('spark.driver.memory', '4g')\n    .config('spark.sql.shuffle.partitions', '200')\n    .getOrCreate())\n\nspark.sparkContext.setLogLevel('WARN')\nsns.set_theme(style='whitegrid')\ndevice

In [ ]:
silver_df = spark.read.parquet(SILVER_PATH)
silver_df.createOrReplaceTempView('trips_silver')

# Split temporal via Spark SQL puro (regra do projeto: data prep em SQL).
# Sample em SQL via TABLESAMPLE (BERNOULLI) e ORDER BY RAND() truncado por LIMIT.
sample_cols_sql = ', '.join(SAMPLE_COLS)
train_spark = spark.sql(f"""
    SELECT {sample_cols_sql}
    FROM trips_silver
    WHERE pickup_datetime < TIMESTAMP '{SPLIT_DATE}'
      AND RAND({SEED}) < {TRAIN_FRACTION}
    LIMIT {MAX_TRAIN_ROWS}
""")
test_spark = spark.sql(f"""
    SELECT {sample_cols_sql}
    FROM trips_silver
    WHERE pickup_datetime >= TIMESTAMP '{SPLIT_DATE}'
      AND RAND({SEED + 1}) < {TEST_FRACTION}
    LIMIT {MAX_TEST_ROWS}
""")

train_pdf = train_spark.toPandas()
test_pdf = test_spark.toPandas()

if train_pdf.empty or test_pdf.empty:
    raise ValueError('A NN exige dados antes e depois de 2023-06-01. Ajuste os arquivos em /data antes de seguir.')

test_context_pdf = test_pdf[RAW_CONTEXT_COLS].copy()
train_pdf.shape, test_pdf.shape

In [ ]:
# Val split aleatorio (nao temporal) e proposital: o test set ja foi separado temporalmente.
# Aqui queremos so estimativa de loss para early stopping; nao se busca temporalidade dupla.
train_pdf, val_pdf = train_test_split(train_pdf, test_size=0.2, random_state=SEED)

train_pdf = pd.get_dummies(train_pdf, columns=CATEGORICAL_COLS, dummy_na=True)
val_pdf = pd.get_dummies(val_pdf, columns=CATEGORICAL_COLS, dummy_na=True)
test_pdf = pd.get_dummies(test_pdf, columns=CATEGORICAL_COLS, dummy_na=True)

feature_columns = [col for col in train_pdf.columns if col != TARGET_COL]
val_pdf = val_pdf.reindex(columns=train_pdf.columns, fill_value=0)
test_pdf = test_pdf.reindex(columns=train_pdf.columns, fill_value=0)

scaler = StandardScaler()
train_pdf.loc[:, NUMERIC_COLS] = scaler.fit_transform(train_pdf[NUMERIC_COLS])
val_pdf.loc[:, NUMERIC_COLS] = scaler.transform(val_pdf[NUMERIC_COLS])
test_pdf.loc[:, NUMERIC_COLS] = scaler.transform(test_pdf[NUMERIC_COLS])

In [ ]:
def frame_to_tensors(frame: pd.DataFrame):\n    x = torch.tensor(frame[feature_columns].astype('float32').to_numpy(), dtype=torch.float32)\n    y = torch.tensor(frame[TARGET_COL].astype('float32').to_numpy().reshape(-1, 1), dtype=torch.float32)\n    return x, y\n\nx_train, y_train = frame_to_tensors(train_pdf)\nx_val, y_val = frame_to_tensors(val_pdf)\nx_test, y_test = frame_to_tensors(test_pdf)\n\ntrain_loader = DataLoader(TensorDataset(x_train, y_train), batch_size=BATCH_SIZE, shuffle=True)\nval_loader = DataLoader(TensorDataset(x_val, y_val), batch_size=BATCH_SIZE, shuffle=False)\ntest_loader = DataLoader(TensorDataset(x_test, y_test), batch_size=BATCH_SIZE, shuffle=False)\n\nclass FareRegressor(nn.Module):\n    def __init__(self, input_dim: int):\n        super().__init__()\n        self.net = nn.Sequential(\n            nn.Linear(input_dim, 128),\n            nn.ReLU(),\n            nn.Dropout(0.2),\n            nn.Linear(128, 64),\n            nn.ReLU(),\n            nn.Dropout(0.2),\n            nn.Linear(64, 32),\n            nn.ReLU(),\n            nn.Linear(32, 1)\n        )\n\n    def forward(self, features):\n        return self.net(features)\n\nmodel = FareRegressor(input_dim=len(feature_columns)).to(device)\nloss_fn = nn.MSELoss()\noptimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [ ]:
best_state = None\nbest_val_loss = float('inf')\nepochs_without_improvement = 0\nhistory = []\n\nstart_time = time.perf_counter()\nfor epoch in range(1, EPOCHS + 1):\n    model.train()\n    train_losses = []\n    for batch_x, batch_y in train_loader:\n        batch_x = batch_x.to(device)\n        batch_y = batch_y.to(device)\n\n        optimizer.zero_grad()\n        preds = model(batch_x)\n        loss = loss_fn(preds, batch_y)\n        loss.backward()\n        optimizer.step()\n        train_losses.append(loss.item())\n\n    model.eval()\n    val_losses = []\n    with torch.no_grad():\n        for batch_x, batch_y in val_loader:\n            batch_x = batch_x.to(device)\n            batch_y = batch_y.to(device)\n            val_losses.append(loss_fn(model(batch_x), batch_y).item())\n\n    avg_train_loss = float(np.mean(train_losses))\n    avg_val_loss = float(np.mean(val_losses))\n    history.append({'epoch': epoch, 'train_loss': avg_train_loss, 'val_loss': avg_val_loss})\n    print(f'Epoch {epoch:02d} | train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}')\n\n    if avg_val_loss < best_val_loss:\n        best_val_loss = avg_val_loss\n        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}\n        epochs_without_improvement = 0\n    else:\n        epochs_without_improvement += 1\n\n    if epochs_without_improvement >= PATIENCE:\n        print('Early stopping acionado.')\n        break\n\ntrain_seconds = time.perf_counter() - start_time\nmodel.load_state_dict(best_state)

In [ ]:
model.eval()
pred_batches = []
with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        pred_batches.append(model(batch_x).cpu().numpy())

y_pred = np.vstack(pred_batches).reshape(-1)
y_true = y_test.numpy().reshape(-1)

# np.sqrt(mean_squared_error) em vez de squared=False (deprecated em sklearn>=1.4, removido em 1.6)
metrics = {
    'rmse': float(np.sqrt(mean_squared_error(y_true, y_pred))),
    'mae': float(mean_absolute_error(y_true, y_pred)),
    'r2': float(r2_score(y_true, y_pred))
}
metrics

In [ ]:
eval_pdf = test_context_pdf.copy()
eval_pdf['prediction'] = y_pred
eval_pdf['abs_error'] = (eval_pdf[TARGET_COL] - eval_pdf['prediction']).abs()
eval_pdf['sq_error'] = (eval_pdf[TARGET_COL] - eval_pdf['prediction']) ** 2
eval_pdf['distance_bucket'] = pd.cut(
    eval_pdf['trip_miles'],
    bins=[-np.inf, 2, 5, 10, np.inf],
    labels=['0-2 mi', '2-5 mi', '5-10 mi', '10+ mi']
)

borough_errors = (eval_pdf.groupby('pu_borough', observed=True)
    .agg(rows=(TARGET_COL, 'size'), mae=('abs_error', 'mean'), rmse=('sq_error', lambda s: np.sqrt(s.mean())))
    .sort_values('rmse', ascending=False))
distance_errors = (eval_pdf.groupby('distance_bucket', observed=True)
    .agg(rows=(TARGET_COL, 'size'), mae=('abs_error', 'mean'), rmse=('sq_error', lambda s: np.sqrt(s.mean()))))

borough_errors, distance_errors

In [ ]:
history_pdf = pd.DataFrame(history)\nresidual_pdf = eval_pdf.copy()\nresidual_pdf['residual'] = residual_pdf[TARGET_COL] - residual_pdf['prediction']\n\nfig, axes = plt.subplots(1, 3, figsize=(18, 5))\nsns.lineplot(data=history_pdf, x='epoch', y='train_loss', ax=axes[0], label='train')\nsns.lineplot(data=history_pdf, x='epoch', y='val_loss', ax=axes[0], label='val')\naxes[0].set_title('Loss por epoca - NN')\nsns.histplot(residual_pdf['residual'], bins=50, ax=axes[1])\naxes[1].set_title('Distribuicao dos residuos - NN')\nsns.scatterplot(data=residual_pdf.sample(min(len(residual_pdf), 10000), random_state=SEED), x='prediction', y='residual', s=10, alpha=0.3, ax=axes[2])\naxes[2].axhline(0, color='black', linestyle='--', linewidth=1)\naxes[2].set_title('Residuo vs predito - NN')\nplt.tight_layout()\nplt.savefig(RESIDUAL_PLOT, dpi=150, bbox_inches='tight')\nplt.show()

In [ ]:
torch.save({\n    'state_dict': model.state_dict(),\n    'feature_columns': feature_columns,\n    'numeric_cols': NUMERIC_COLS,\n    'sample_rows_train': len(train_pdf),\n    'sample_rows_test': len(test_pdf)\n}, MODEL_OUTPUT)\n\nresults_df = pd.read_csv(RESULTS_PATH) if RESULTS_PATH.exists() else pd.DataFrame(columns=['model', 'rmse', 'mae', 'r2', 'train_seconds', 'notes'])\nresults_df = results_df[results_df['model'] != 'neural_network']\nresults_df = pd.concat([\n    results_df,\n    pd.DataFrame([{\n        'model': 'neural_network',\n        'rmse': metrics['rmse'],\n        'mae': metrics['mae'],\n        'r2': metrics['r2'],\n        'train_seconds': train_seconds,\n        'notes': f'PyTorch em amostra train={len(train_pdf)} test={len(test_pdf)}'\n    }])\n], ignore_index=True)\nresults_df.to_csv(RESULTS_PATH, index=False)\nresults_df

In [ ]:
spark.stop()